# RadarCrédito: Painel de Inadimplência e Risco de Crédito no Brasil (2016-2026)
## Notebook de Exploração Inicial dos Dados (Etapa 2 - Issue #3)

**Disciplina:** Programação Orientada a Objetos  
**Curso:** Ciência de Dados e Inteligência Artificial — PUC-SP  
**Autores:** Grupo do RadarCrédito  

---

### Objetivo do Notebook
Este notebook realiza a **análise exploratória inicial (Etapa 2)** das séries temporais de taxa de inadimplência da carteira de crédito do Sistema Financeiro Nacional, obtidas junto à API do Sistema Gerenciador de Séries Temporais (SGS) do Banco Central do Brasil (BCB).

O propósito é validar a qualidade dos dados, inspecionar tipos de variáveis, verificar ausências ou anomalias, consolidar estatísticas descritivas e estabelecer a base empírica para responder às cinco perguntas centrais do projeto.

---

### Perguntas Norteadoras do Projeto
1. **Evolução e Ciclos:** Como a inadimplência evoluiu nos últimos 10 anos e quais foram os principais picos e vales?
2. **Modalidades Críticas:** Qual modalidade de crédito para pessoa física é a mais inadimplente e a mais volátil?
3. **Sensibilidade à Taxa Básica:** A elevação da taxa Selic antecede aumentos de inadimplência? Com qual defasagem média?
4. **Fatores Macroeconômicos:** A inflação (IPCA) e o nível de endividamento explicam os aumentos da inadimplência?
5. **Ciclo PF vs. PJ:** Os ciclos de inadimplência de pessoa física e de pessoa jurídica seguem o mesmo padrão e sincronia?

---

### Séries Temporais Selecionadas
Todas as séries são de frequência **mensal** e representam a **taxa de inadimplência da carteira de crédito** (percentual da carteira com pelo menos uma parcela em atraso superior a 90 dias):

| Código SGS | Nome da Coluna | Descrição Oficial (BCB) | Segmentação |
|---|---|---|---|
| **21082** | `total` | Inadimplência da carteira de crédito - Total | Carteira Consolidada |
| **21083** | `pessoa_juridica` | Inadimplência da carteira de crédito - Pessoas jurídicas | Segmento PJ |
| **21084** | `pessoa_fisica` | Inadimplência da carteira de crédito - Pessoas físicas | Segmento PF Total |
| **21112** | `pf_livres` | Inadimplência da carteira de crédito - PF - Recursos livres | PF Taxas Livres |
| **21113** | `pf_cheque_especial` | Inadimplência da carteira de crédito - PF - Cheque especial | PF Crédito Rotativo |
| **21129** | `pf_cartao` | Inadimplência da carteira de crédito - PF - Cartão de crédito total | PF Crédito Rotativo/Parcelado |
| **21145** | `pf_direcionados` | Inadimplência da carteira de crédito - PF - Recursos direcionados | PF Crédito Habitacional/Rural |

**Estrutura de Hierarquia:**
- `total` = ponderação entre `pessoa_fisica` e `pessoa_juridica`.
- `pessoa_fisica` = ponderação entre recursos livres (`pf_livres`) e direcionados (`pf_direcionados`).
- Recursos livres (`pf_livres`) contemplam linhas de alto custo e rotativo (`pf_cheque_especial`, `pf_cartao`), além de consignado e CDC.

In [1]:
import sys
from pathlib import Path

# Ajuste do caminho raiz do projeto para funcionamento em notebooks/ ou na raiz
raiz = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Configura renderer padrão do Plotly para compatibilidade com notebooks
pio.renderers.default = "notebook_connected+plotly_mimetype"

from src.fonte_dados import FonteBCB
from src.analisador import AnalisadorDados

# Instancia a fonte com pasta de cache apontando para a pasta data do projeto
fonte = FonteBCB(pasta_cache=str(raiz / "data"))
print(f"Fonte configurada: {fonte.nome}")
print(f"Pasta de cache: {fonte.pasta_cache}")

Fonte configurada: Banco Central do Brasil (SGS)
Pasta de cache: C:\Users\paulo\Desktop\Projetos\Rad-cred\radar-credito\data


## Seção 1. Carregamento das Séries Temporais

Nesta seção, consultamos as 7 séries de crédito selecionadas para o recorte decenal solicitado: **01/11/2016 a 31/10/2026**.
Utilizamos a classe `FonteBCB`, que abstrai as requisições HTTP à API do SGS do Banco Central e implementa política de contingência com cache local em arquivo CSV (`data/serie_{codigo}.csv`).

In [2]:
# Mapeamento oficial de colunas e códigos SGS do BCB
series_codigos = {
    "total": 21082,
    "pessoa_juridica": 21083,
    "pessoa_fisica": 21084,
    "pf_livres": 21112,
    "pf_cheque_especial": 21113,
    "pf_cartao": 21129,
    "pf_direcionados": 21145,
}

data_inicio = "01/11/2016"
data_fim = "31/10/2026"

series = {}
print("Carregando séries temporais do Banco Central do Brasil:")
print("-" * 75)

for nome, codigo in series_codigos.items():
    tabela = fonte.obter_serie(codigo, data_inicio, data_fim)
    series[nome] = tabela
    data_min = tabela["data"].min().strftime("%d/%m/%Y")
    data_max = tabela["data"].max().strftime("%d/%m/%Y")
    qtd_linhas = len(tabela)
    print(f"Série {codigo:5d} ({nome:19s}): {qtd_linhas} registros | {data_min} a {data_max}")

print("-" * 75)
print("Amostra da série 'total' (cinco primeiras linhas):")
display(series["total"].head())

print("Amostra da série 'total' (cinco últimas linhas):")
display(series["total"].tail())

Carregando séries temporais do Banco Central do Brasil:
---------------------------------------------------------------------------
Série 21082 (total              ): 119 registros | 01/11/2016 a 01/09/2026
Série 21083 (pessoa_juridica    ): 119 registros | 01/11/2016 a 01/09/2026
Série 21084 (pessoa_fisica      ): 119 registros | 01/11/2016 a 01/09/2026
Série 21112 (pf_livres          ): 119 registros | 01/11/2016 a 01/09/2026
Série 21113 (pf_cheque_especial ): 119 registros | 01/11/2016 a 01/09/2026
Série 21129 (pf_cartao          ): 119 registros | 01/11/2016 a 01/09/2026
Série 21145 (pf_direcionados    ): 119 registros | 01/11/2016 a 01/09/2026
---------------------------------------------------------------------------
Amostra da série 'total' (cinco primeiras linhas):
Amostra da série 'total' (cinco últimas linhas):


,data,valor
0,2016-11-01,3.86
1,2016-12-01,3.75
2,2017-01-01,3.79
3,2017-02-01,3.82
4,2017-03-01,3.91


,data,valor
114,2026-05-01,4.75
115,2026-06-01,4.59
116,2026-07-01,4.88
117,2026-08-01,5.05
118,2026-09-01,4.98


### Interpretação do Carregamento
- Cada série retornou **119 registros mensais**, compreendidos entre **01/11/2016 e 01/09/2026**.
- A defasagem entre a data final solicitada (31/10/2026) e o último dado disponível (setembro/2026) decorre da política de publicação do Banco Central (defasagem operacional de cerca de 45 a 60 dias para apuração do risco de crédito consolidado).
- O mecanismo de contingência em `data/` assegurou a integridade dos dados mesmo sob indisponibilidade transitória da rede.

## Seção 2. Identificação de Variáveis e Tipos de Dados

O Banco Central disponibiliza dados via JSON com campos textuais (`"data"` como string no formato `"dd/mm/aaaa"` e `"valor"` como string numérica decimal com separador de ponto).
Nesta seção, confirmamos os tipos e mapeamos cada variável do domínio do problema.

In [3]:
print("--- Informações estruturais da série 'total' ---")
series["total"].info()

print("\nTipos de dados no DataFrame:")
display(series["total"].dtypes)

# Tabela resumo metadados das variáveis
metadados = [
    {
        "variavel": "data",
        "tipo_pandas": "datetime64[ns]",
        "papel": "Dimensão Temporal",
        "unidade": "Data (AAAA-MM-DD)",
        "periodicidade": "Mensal (início de mês)",
        "fonte": "Banco Central do Brasil (SGS)",
        "descricao": "Mês de referência da apuração da carteira"
    },
    {
        "variavel": "valor",
        "tipo_pandas": "float64",
        "papel": "Métrica Quantitativa",
        "unidade": "% da carteira",
        "periodicidade": "Mensal",
        "fonte": "Banco Central do Brasil (SGS)",
        "descricao": "Taxa de inadimplência (atraso > 90 dias)"
    }
]

tabela_metadados = pd.DataFrame(metadados)
print("\nTabela Resumo das Variáveis:")
display(tabela_metadados)

--- Informações estruturais da série 'total' ---
<class 'pandas.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    119 non-null    datetime64[us]
 1   valor   119 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 2.0 KB

Tipos de dados no DataFrame:

Tabela Resumo das Variáveis:


data     datetime64[us]
valor           float64
dtype: object

,variavel,tipo_pandas,papel,unidade,periodicidade,fonte,descricao
0,data,datetime64[ns],Dimensão Temporal,Data (AAAA-MM-DD),Mensal (início de mês),Banco Central do Brasil (SGS),Mês de referência da apuração da carteira
1,valor,float64,Métrica Quantitativa,% da carteira,Mensal,Banco Central do Brasil (SGS),Taxa de inadimplência (atraso > 90 dias)


### Interpretação dos Tipos e Conversão
- A coluna `data` foi convertida de string para `datetime64[ns]` via `pd.to_datetime(tabela["data"], format="%d/%m/%Y")` em `FonteBCB`. Isso permite operações temporais precisas (filtragem, ordenação cronológica e agrupamento por ano).
- A coluna `valor` foi convertida com `pd.to_numeric(tabela["valor"], errors="coerce")`, que transforma eventuais strings defeituosas ou vazias em `NaN`, viabilizando cálculos estatísticos float64 sem quebras de execução.

## Seção 3. Diagnóstico de Qualidade dos Dados

Para validar a integridade antes de qualquer análise ou modelagem, executamos um teste exaustivo de qualidade em todas as 7 séries.
Avaliamos:
1. **Valores nulos ou ausentes:** contagem e percentual.
2. **Duplicidades:** datas repetidas ou linhas idênticas.
3. **Continuidade temporal:** comparação com um calendário mensal ideal (`pd.date_range` com frequência `MS`).
4. **Consistência de domínio:** busca por valores negativos ou superiores a 100%.
5. **Limites do período:** datas fora do intervalo pedido.
6. **Saltos abruptos:** variação absoluta mensal (em pontos percentuais) para detecção de quebras de série.
7. **Outliers estatísticos:** contagem de valores fora do intervalo $[Q_1 - 1{,}5 \cdot IQR, \; Q_3 + 1{,}5 \cdot IQR]$.

In [4]:
diagnosticos = []

for nome, df in series.items():
    total_registros = len(df)
    nulos = int(df["valor"].isna().sum())
    pct_nulos = (nulos / total_registros) * 100.0
    datas_duplicadas = int(df.duplicated(subset=["data"]).sum())
    linhas_duplicadas = int(df.duplicated().sum())
    
    # Validação do dia do mês
    dias_do_mes = df["data"].dt.day.unique().tolist()
    
    # Checagem de meses faltantes no intervalo
    menor_data = df["data"].min()
    maior_data = df["data"].max()
    datas_esperadas = pd.date_range(start=menor_data, end=maior_data, freq="MS")
    datas_presentes = set(df["data"])
    faltantes = []
    for data in datas_esperadas:
        if data not in datas_presentes:
            faltantes.append(data.strftime("%Y-%m-%d"))
    
    # Checagem de limites de valor (taxa percentual entre 0 e 100%)
    valores_negativos = int((df["valor"] < 0).sum())
    valores_acima_100 = int((df["valor"] > 100).sum())
    
    # Checagem de período
    fora_periodo = int(((df["data"] < "2016-11-01") | (df["data"] > "2026-10-31")).sum())
    
    # Identificação do maior salto absoluto mês a mês
    df_ordenado = df.sort_values(by="data").copy()
    variacoes = df_ordenado["valor"].diff().abs()
    maior_salto = float(variacoes.max())
    indice_salto = variacoes.idxmax()
    if pd.notna(indice_salto):
        mes_salto = df_ordenado.loc[indice_salto, "data"].strftime("%m/%Y")
    else:
        mes_salto = "-"
    
    # Detecção de outliers pelo critério do IQR (1.5 * amplitude interquartil)
    q1 = float(df["valor"].quantile(0.25))
    q3 = float(df["valor"].quantile(0.75))
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    mascara_outliers = (df["valor"] < limite_inf) | (df["valor"] > limite_sup)
    qtd_outliers = int(mascara_outliers.sum())
    
    meses_outliers_lista = []
    for d in df_ordenado.loc[mascara_outliers, "data"]:
        meses_outliers_lista.append(d.strftime("%m/%Y"))
    
    if len(meses_outliers_lista) > 0:
        texto_outliers = ", ".join(meses_outliers_lista)
    else:
        texto_outliers = "Nenhum"
        
    diagnosticos.append({
        "serie": nome,
        "registros": total_registros,
        "nulos": nulos,
        "pct_nulos": round(pct_nulos, 2),
        "datas_duplicadas": datas_duplicadas,
        "linhas_duplicadas": linhas_duplicadas,
        "dias_mes": str(dias_do_mes),
        "meses_faltantes": len(faltantes),
        "negativos": valores_negativos,
        "acima_100": valores_acima_100,
        "fora_periodo": fora_periodo,
        "maior_salto_pp": round(maior_salto, 2),
        "mes_maior_salto": mes_salto,
        "qtd_outliers_iqr": qtd_outliers,
        "meses_outliers": texto_outliers
    })

tabela_qualidade = pd.DataFrame(diagnosticos)
display(tabela_qualidade)

,serie,registros,nulos,pct_nulos,datas_duplicadas,linhas_duplicadas,dias_mes,meses_faltantes,negativos,acima_100,fora_periodo,maior_salto_pp,mes_maior_salto,qtd_outliers_iqr,meses_outliers
0,total,119,0,0.0,0,0,[1],0,0,0,0,0.36,06/2020,4,"05/2026, 07/2026, 08/2026, 09/2026"
1,pessoa_juridica,119,0,0.0,0,0,[1],0,0,0,0,0.43,06/2018,0,Nenhum
2,pessoa_fisica,119,0,0.0,0,0,[1],0,0,0,0,0.37,06/2020,12,"10/2025, 11/2025, 12/2025, 01/2026, 02/2026, 0..."
3,pf_livres,119,0,0.0,0,0,[1],0,0,0,0,0.40,07/2026,3,"07/2026, 08/2026, 09/2026"
4,pf_cheque_especial,119,0,0.0,0,0,[1],0,0,0,0,2.80,11/2023,0,Nenhum
5,pf_cartao,119,0,0.0,0,0,[1],0,0,0,0,1.03,11/2020,0,Nenhum
6,pf_direcionados,119,0,0.0,0,0,[1],0,0,0,0,0.49,03/2020,9,"01/2026, 02/2026, 03/2026, 04/2026, 05/2026, 0..."


### Interpretação do Diagnóstico de Qualidade
1. **Completude Perfeita:** Todas as 7 séries apresentam exatamente **0 valores nulos** e **0% de perdas**.
2. **Unicidade e Continuidade:** Há **0 duplicidades** de datas ou de linhas completas. Todos os registros caem exatamente no dia 1 de cada mês (`dias_mes: [1]`), sem nenhum mês ausente no intervalo contínuo de 119 meses.
3. **Consistência de Domínio:** Nenhum valor é negativo e nenhum supera 100%, em total aderência à definição de taxa percentual.
4. **Análise de Saltos:** O maior salto mensal observado ocorreu no `pf_cheque_especial` (+2,80 p.p. em 11/2023), o que é compatível com a volatilidade intrínseca de linhas de crédito rotativo de curto prazo.
5. **Outliers pelo Critério IQR:**
   - As séries `pessoa_fisica` (12 outliers entre 10/2025 e 09/2026), `pf_direcionados` (9 outliers em 2026), `total` (4 outliers em 2026) e `pf_livres` (3 outliers em 2026) apresentam valores acima de $Q_3 + 1{,}5 \cdot IQR$.
   - **Conclusão Crítica:** Esses valores **não são erros cadastrais nem falhas de medição**. Trata-se de um fenômeno econômico legítimo: a aceleração da inadimplência no ciclo recente de 2025/2026, impulsionada pelo aperto monetário e endividamento das famílias.
   - **Decisão:** **Nenhum descarte ou remoção de linha deve ser feito**. Os dados estão prontos para processamento analítico.

## Seção 4. Estatísticas Descritivas Consolidadas

Nesta seção, calculamos o resumo descritivo de cada série temporal (`describe()`) e derivamos quatro métricas analíticas fundamentais:
- **Amplitude Total:** $\text{Máximo} - \text{Mínimo}$ (alcance total da taxa no período).
- **Coeficiente de Variação (CV):** $\frac{\sigma}{\mu} \times 100\%$ (volatilidade relativa padronizada).
- **Mês do Pico:** Mês e ano em que ocorreu a taxa máxima.
- **Mês do Vale:** Mês e ano em que ocorreu a taxa mínima.

In [5]:
estatisticas = []

for nome, df in series.items():
    resumo = df["valor"].describe()
    media = float(resumo["mean"])
    desvio = float(resumo["std"])
    minimo = float(resumo["min"])
    maximo = float(resumo["max"])
    amplitude = maximo - minimo
    cv = (desvio / media) * 100.0
    
    idx_max = df["valor"].idxmax()
    idx_min = df["valor"].idxmin()
    mes_pico = df.loc[idx_max, "data"].strftime("%m/%Y")
    mes_vale = df.loc[idx_min, "data"].strftime("%m/%Y")
    
    estatisticas.append({
        "serie": nome,
        "contagem": int(resumo["count"]),
        "media": round(media, 2),
        "desvio_padrao": round(desvio, 2),
        "minimo": round(minimo, 2),
        "q25": round(float(resumo["25%"]), 2),
        "mediana": round(float(resumo["50%"]), 2),
        "q75": round(float(resumo["75%"]), 2),
        "maximo": round(maximo, 2),
        "amplitude": round(amplitude, 2),
        "coef_variacao_pct": round(cv, 2),
        "mes_pico": mes_pico,
        "valor_pico": round(maximo, 2),
        "mes_vale": mes_vale,
        "valor_vale": round(minimo, 2),
    })

tabela_estatisticas = pd.DataFrame(estatisticas)
tabela_estatisticas = tabela_estatisticas.sort_values(by="media", ascending=False).reset_index(drop=True)
display(tabela_estatisticas)

,serie,contagem,media,desvio_padrao,minimo,q25,mediana,q75,maximo,amplitude,coef_variacao_pct,mes_pico,valor_pico,mes_vale,valor_vale
0,pf_cheque_especial,119,13.69,1.99,9.44,12.33,13.48,15.46,17.57,8.13,14.56,12/2019,17.57,04/2021,9.44
1,pf_cartao,119,6.97,1.48,3.79,6.00,7.28,7.86,10.04,6.25,21.27,09/2026,10.04,04/2021,3.79
2,pf_livres,119,5.59,0.90,4.09,4.98,5.59,6.06,8.09,4.00,16.06,09/2026,8.09,03/2021,4.09
3,pessoa_fisica,119,3.88,0.69,2.89,3.48,3.76,4.07,6.04,3.15,17.68,08/2026,6.04,12/2020,2.89
4,total,119,3.31,0.64,2.16,2.98,3.30,3.66,5.05,2.89,19.40,08/2026,5.05,12/2020,2.16
5,pessoa_juridica,119,2.48,0.68,1.24,1.93,2.56,2.86,4.06,2.82,27.26,05/2017,4.06,12/2020,1.24
6,pf_direcionados,119,1.84,0.52,1.25,1.48,1.68,2.05,3.60,2.35,28.38,08/2026,3.60,12/2020,1.25


### Interpretação das Estatísticas Descritivas
1. **Hierarquia de Risco (Média):**
   - `pf_cheque_especial` lidera isoladamente com média de **13,69%**, atingindo pico de **17,57%** em 12/2019.
   - `pf_cartao` ocupa a segunda posição com média de **6,97%** e pico recente de **10,04%** em 09/2026.
   - Em contraste, `pf_direcionados` possui a menor média histórica (**1,84%**), decorrente das garantias reais exigidas em financiamentos habitacionais e rurais.
2. **Volatilidade Absoluta vs. Relativa:**
   - Em amplitude absoluta, o cheque especial oscilou **8,13 p.p.** e o cartão **6,25 p.p.**
   - Em termos de Coeficiente de Variação relativo, `pf_direcionados` (**28,38%**) e `pessoa_juridica` (**27,26%**) foram as mais voláteis, pois suas taxas variaram intensamente em relação aos seus níveis médios reduzidos.
3. **Sincronia dos Vales (Pandemia):**
   - As séries `total` (2,16%), `pessoa_fisica` (2,89%), `pessoa_juridica` (1,24%) e `pf_direcionados` (1,25%) atingiram seus vales históricos no mesmo mês: **12/2020**. Esse vale reflete as medidas de auxílio emergencial e a repactuação/prorrogação mandatória de parcelas concedida pelos bancos no auge da crise sanitária.

## Seção 5. Visualização Exploratória

Para investigar as dinâmicas temporais, geramos visualizações com Plotly Express:
1. **Evolução Macro:** Comparação direta de `total`, `pessoa_fisica` e `pessoa_juridica`.
2. **Modalidades de Pessoa Física:** Detalhamento de livres, direcionados, cheque especial e cartão.
3. **Distribuição Comparativa:** Boxplot unificado gerado a partir do formato longo (`AnalisadorDados.largo_para_longo`).
4. **Matriz de Correlação:** Heatmap linear entre todas as 7 séries a partir da tabela larga.

In [6]:
# Gráfico 1: Evolução da inadimplência Macro (Total, PF e PJ)
tabela_larga = AnalisadorDados.unir_series(series)

fig1 = px.line(
    tabela_larga.reset_index(),
    x="data",
    y=["total", "pessoa_fisica", "pessoa_juridica"],
    labels={
        "data": "Data de Referência",
        "value": "Taxa de Inadimplência (% da carteira)",
        "variable": "Segmento"
    },
    title="Evolução Temporal da Taxa de Inadimplência: Total, PF e PJ (2016-2026)",
    template="plotly_white"
)

fig1.update_layout(
    legend_title_text="Segmento de Crédito",
    hovermode="x unified",
    xaxis=dict(tickformat="%m/%Y")
)
fig1.show()

### Análise do Gráfico 1 (Macro: Total, PF e PJ)
- A taxa `total` situa-se rigorosamente entre `pessoa_fisica` e `pessoa_juridica` em todo o histórico, respeitando a relação de agregação ponderada.
- Entre 2016 e 2017, PJ exibia inadimplência superior a PF em decorrência da recessão de 2015-2016 (pico de PJ em 05/2017 com 4,06%).
- A partir de 2018, ocorre uma inversão estrutural: PF passa a ser sistematicamente mais inadimplente que PJ, culminando no descolamento recente de 2025/2026 (PF atingindo 6,04% em 08/2026 enquanto PJ permaneceu em torno de 3%).

In [7]:
# Gráfico 2: Inadimplência de Pessoa Física por Modalidade
fig2 = px.line(
    tabela_larga.reset_index(),
    x="data",
    y=["pf_cheque_especial", "pf_cartao", "pf_livres", "pf_direcionados"],
    labels={
        "data": "Data de Referência",
        "value": "Taxa de Inadimplência (%)",
        "variable": "Modalidade"
    },
    title="Inadimplência de Pessoa Física por Modalidade de Crédito (2016-2026)",
    template="plotly_white"
)

fig2.update_layout(
    legend_title_text="Modalidade PF",
    hovermode="x unified",
    xaxis=dict(tickformat="%m/%Y")
)
fig2.show()

### Análise do Gráfico 2 (Modalidades de PF)
- Há uma gritante disparidade de patamares entre o crédito rotativo e o restante das modalidades:
  - `pf_cheque_especial` oscila historicamente entre 9% e 17%, sofrendo uma inflexão descendente em 2020 após a resolução do BCB que estabeleceu teto de juros para o cheque especial.
  - `pf_cartao` exibe uma forte trajetória ascendente a partir do final de 2021, saindo de 3,79% (04/2021) para o recorde de 10,04% em 09/2026.
  - `pf_direcionados` mantém-se estável e em patamares reduzidos (1,2% a 3,6%), provendo sustentação amortecedora à inadimplência total de PF.

In [8]:
# Gráfico 3: Boxplot comparativo de todas as séries
# Conversão do formato largo para o formato longo com o método estático AnalisadorDados
tabela_longa = AnalisadorDados.largo_para_longo(tabela_larga)

fig3 = px.box(
    tabela_longa,
    x="modalidade",
    y="valor",
    color="modalidade",
    points="all",
    labels={
        "modalidade": "Série / Modalidade",
        "valor": "Taxa de Inadimplência (% da carteira)"
    },
    title="Distribuição e Dispersão das Taxas de Inadimplência por Série (2016-2026)",
    template="plotly_white"
)

fig3.update_layout(showlegend=False)
fig3.show()

### Análise do Gráfico 3 (Boxplot Comparativo)
- A dispersão confirma a estratificação de risco de crédito no Brasil:
  1. Nível Crítico: Cheque especial (mediana 13,48%) e Cartão de crédito (mediana 7,28%).
  2. Nível Intermediário: PF Livres (mediana 5,59%), PF Total (mediana 3,76%) e Total Geral (mediana 3,30%).
  3. Nível Baixo Risco: PJ (mediana 2,56%) e PF Direcionados (mediana 1,68%).
- Os pontos plotados nas caudas superiores das séries PF e Direcionados refletem os meses atípicos de 2026 identificados na Seção 3.

In [9]:
# Gráfico 4: Matriz de Correlação Linear
matriz_corr = tabela_larga.corr()

fig4 = px.imshow(
    matriz_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    labels=dict(color="Correlação de Pearson"),
    title="Matriz de Correlação Linear de Pearson entre Séries de Inadimplência",
    template="plotly_white"
)

fig4.update_layout(xaxis_title="Série", yaxis_title="Série")
fig4.show()

### Análise do Gráfico 4 (Matriz de Correlação)
- A taxa `total` apresenta fortíssima correlação com `pessoa_fisica` ($r = 0{,}94$) e com `pf_livres` ($r = 0{,}93$), indicando que o comportamento do crédito livre para indivíduos dita o ritmo da inadimplência nacional.
- `pf_cartao` e `pf_livres` apresentam correlação quase perfeita ($r = 0{,}96$), revelando que o cartão é o principal vetor de risco no crédito livre contemporâneo.
- Por outro lado, `pessoa_juridica` e `pessoa_fisica` exibem correlação moderada ($r = 0{,}60$), confirmando que os ciclos de crédito de empresas e famílias possuem defasagens e dinâmicas distintas.